# Black–Scholes: prices, greeks, put–call parity, American options

`tq.BlackScholesPricer` on TICKER1 (discrete dividends): call/put prices and greeks by AAD, put–call parity, greeks as
functions of spot, time and volatility, European vs American (Bjerksund–Stensland 1993).
Docs: *Models and Pricing* (Black–Scholes), *Exercises* (`equity.tex`, Options and Black–Scholes).

In [ ]:
import os, sys
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq


## Market

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
eq = mkt.get_equity("TICKER1")
pricer = tq.BlackScholesPricer(dividend_model="discrete")

EXPIRY = EVALUATION_DATE + timedelta(days=365)
S0 = float(eq.spot)
K = round(S0, 2)   # ATM
eq.summary(EXPIRY)

## Prices and greeks (AAD)
One tape on the market risk factors; gamma from a second tape around the first.

| greek | definition | unit |
|---|---|---|
| delta, gamma | $\partial V/\partial S$, $\partial^2 V/\partial S^2$ | — |
| vega | $\sum_{nodes} \partial V/\partial\sigma_{ij}$ (parallel vol shift) | per 1 vol point |
| theta | $-\partial V/\partial T$ | per day |
| rho | $\sum_{pillars} \partial V/\partial r_i$ (parallel zero-rate shift) | per bp |

In [ ]:
def option(option_type, strike=K, expiry=EXPIRY):
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry,
                            option_type=option_type, strike=strike, underlying=eq.ticker,
                            exercise_type=tq.ExerciseType.European)

def greeks(opt):
    with tf.GradientTape() as outer:
        pricer.price(opt, mkt, autodiff=True)
        d_spot, d_vol, d_t, d_curve = pricer.tape.gradient(
            opt.price, [eq.spot, eq.vol_surface.volatility_matrix, opt.time_to_maturity, eq.discount_curve.variables])
    return {
        "price": float(opt.price),
        "delta": float(d_spot),
        "gamma": float(outer.gradient(d_spot, eq.spot)),
        "vega":  float(tf.reduce_sum(d_vol)) / 100,
        "theta": -float(d_t) / 365,
        "rho":   sum(float(g) for g in d_curve if g is not None) / 1e4,
    }

call, put = option(tq.OptionType.Call), option(tq.OptionType.Put)
table = pd.DataFrame({"call": greeks(call), "put": greeks(put)})
table

Vega is a map on the surface nodes: with bilinear interpolation only the 4 nodes around $(T, K)$ carry risk (here $K = 1.03\,S_0$, $T \approx 0.82$, off the grid).

In [ ]:
off_grid = option(tq.OptionType.Call, round(1.03 * S0, 2), EVALUATION_DATE + timedelta(days=300))
pricer.price(off_grid, mkt, autodiff=True)
vega_map = pricer.tape.gradient(off_grid.price, eq.vol_surface.volatility_matrix).numpy() / 100
pd.DataFrame(vega_map, index=[f"T={t:.2f}" for t in eq.vol_surface.maturity],
             columns=[f"K={k / S0:.0%}" for k in eq.vol_surface.strike]).round(4)

## Put–call parity
$C - P = P(0,T)\,(F - K)$, model-free. Differentiating: $\Delta_C - \Delta_P = P(0,T)\,\partial F/\partial S = e^{-q_{repo}T}$, $\Gamma_C = \Gamma_P$, $\mathcal{V}_C = \mathcal{V}_P$.

In [ ]:
DF = float(eq.discount_curve.discount(EXPIRY))
rows = []
for k in np.round(S0 * np.array([0.8, 0.9, 1.0, 1.1, 1.2]), 2):
    c, p = option(tq.OptionType.Call, k), option(tq.OptionType.Put, k)
    pricer.price(c, mkt); pricer.price(p, mkt)
    lhs, rhs = float(c.price) - float(p.price), DF * (float(eq.forward(EXPIRY)) - k)
    rows.append({"K": k, "C - P": lhs, "DF (F - K)": rhs, "error": lhs - rhs})
pd.DataFrame(rows).set_index("K")

In [ ]:
T = eq.year_fraction(EXPIRY)
print(f"delta_C - delta_P = {table.loc['delta', 'call'] - table.loc['delta', 'put']:.10f}   e^(-repo T) = {np.exp(-float(eq.repo) * T):.10f}")
print(f"gamma_C - gamma_P = {table.loc['gamma', 'call'] - table.loc['gamma', 'put']:.2e}")
print(f"vega_C  - vega_P  = {table.loc['vega', 'call'] - table.loc['vega', 'put']:.2e}")

## Greeks vs spot
The market spot variable is moved with `assign` and the option repriced.

In [ ]:
def scan(set_value, values, make_options):
    out = {name: [] for name in make_options}
    for v in values:
        set_value(v)
        for name, make in make_options.items():
            out[name].append(greeks(make()))
    return {name: pd.DataFrame(rows, index=values) for name, rows in out.items()}

def plot_greeks(results, xlabel, names=("delta", "gamma", "vega", "theta")):
    fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 3.5))
    for ax, g in zip(axes, names):
        for label, df in results.items():
            ax.plot(df.index, df[g], label=label)
        ax.set_title(g); ax.set_xlabel(xlabel); ax.grid(alpha=0.3)
    axes[0].legend(); plt.tight_layout()

s0 = float(eq.spot)
spots = s0 * np.linspace(0.6, 1.4, 41)
by_spot = scan(eq.spot.assign, spots,
               {"call": lambda: option(tq.OptionType.Call), "put": lambda: option(tq.OptionType.Put)})
eq.spot.assign(s0)
plot_greeks(by_spot, "spot")

## Greeks vs time to maturity
ATM, ITM and OTM calls: gamma and theta concentrate on the ATM strike as expiry approaches. Kinks: ex-dividend dates (discrete dividends enter the forward) and maturity nodes of the vol surface (linear interpolation in $T$).

In [ ]:
days = np.arange(7, 2 * 365 + 1, 7)
expiry_holder = {}
by_time = scan(lambda d: expiry_holder.update(expiry=EVALUATION_DATE + timedelta(days=int(d))), days,
               {f"K={m:.0%}": (lambda k=round(m * S0, 2): option(tq.OptionType.Call, k, expiry_holder["expiry"]))
                for m in (0.8, 1.0, 1.2)})
for df in by_time.values():
    df.index = df.index / 365
plot_greeks(by_time, "T (years)")

## Greeks vs volatility
Parallel shift of the whole vol surface (all nodes), ATM, ITM and OTM calls.

In [ ]:
base = eq.vol_surface.volatility_matrix.numpy().copy()
shifts = np.linspace(-0.15, 0.30, 46)
by_vol = scan(lambda d: eq.vol_surface.volatility_matrix.assign(base + d), shifts,
              {f"K={m:.0%}": (lambda k=round(m * S0, 2): option(tq.OptionType.Call, k)) for m in (0.8, 1.0, 1.2)})
eq.vol_surface.volatility_matrix.assign(base)
atm_vol = float(eq.vol_surface.volatility(K, T))
for df in by_vol.values():
    df.index = (atm_vol + df.index) * 100
plot_greeks(by_vol, "ATM vol (%) after the parallel shift")

## European vs American
American options by the Bjerksund–Stensland (1993) approximation, carry $b = r - q$. Early-exercise premium
EEP = American − European, 1Y, both dividend models: escrowed (carry $r - q_{repo} \approx r > 0$) → the call is never
exercised early, the put is; equivalent yield ($q_{eff} > r$) → the opposite.

In [ ]:
def price_of(option_type, strike, exercise, model, expiry=EXPIRY):
    opt = tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry, option_type=option_type,
                           strike=strike, underlying=eq.ticker, exercise_type=exercise)
    tq.BlackScholesPricer(dividend_model=model).price(opt, mkt)
    return float(opt.price)

rows = []
for model in ("discrete", "continuous"):
    for ot in (tq.OptionType.Call, tq.OptionType.Put):
        for m in (0.8, 0.9, 1.0, 1.1, 1.2):
            k = round(m * S0, 2)
            eu = price_of(ot, k, tq.ExerciseType.European, model)
            am = price_of(ot, k, tq.ExerciseType.American, model)
            rows.append({"model": model, "type": ot.name, "K/S0": m, "European": eu, "American": am, "EEP": am - eu,
                         "EEP %": 100 * (am - eu) / eu})
pd.DataFrame(rows).set_index(["model", "type", "K/S0"]).round(4)

The premium grows with the maturity and jumps at each ex-date entering the option life: the cash dividend lowers the escrowed spot (put) or raises the equivalent yield (call). Deep ITM puts exercise first: the interest on $K$ beats the time value.

In [ ]:
mats = np.arange(30, 3 * 365 + 1, 30)
eep = {}
for label, ot, model in (("put, discrete", tq.OptionType.Put, "discrete"), ("call, continuous", tq.OptionType.Call, "continuous")):
    eep[label] = [price_of(ot, K, tq.ExerciseType.American, model, EVALUATION_DATE + timedelta(days=int(d)))
                  - price_of(ot, K, tq.ExerciseType.European, model, EVALUATION_DATE + timedelta(days=int(d))) for d in mats]
plt.figure(figsize=(7, 4))
for label, v in eep.items():
    plt.plot(mats / 365, v, label=label)
plt.title("ATM early-exercise premium"); plt.xlabel("T (years)"); plt.grid(alpha=0.3); plt.legend();

American put delta (AAD through the approximation) vs the European one: it reaches $-1$ below the exercise boundary,
where the put is worth $K - S$.

In [ ]:
def put_delta(exercise, spot):
    eq.spot.assign(spot)
    opt = tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=EXPIRY, option_type=tq.OptionType.Put,
                           strike=K, underlying=eq.ticker, exercise_type=exercise)
    p = tq.BlackScholesPricer(dividend_model="discrete")
    p.price(opt, mkt, autodiff=True)
    return float(p.tape.gradient(opt.price, eq.spot))

spots_am = s0 * np.linspace(0.5, 1.3, 33)
deltas = {ex.name: [put_delta(ex, s) for s in spots_am] for ex in (tq.ExerciseType.European, tq.ExerciseType.American)}
eq.spot.assign(s0)
plt.figure(figsize=(7, 4))
for name, d in deltas.items():
    plt.plot(spots_am / s0, d, label=name)
plt.title("ATM put delta, discrete dividends"); plt.xlabel("S / S0"); plt.grid(alpha=0.3); plt.legend();

## Exercises
1. Why does $\Delta_C - \Delta_P$ equal $e^{-q_{repo}T}$ and not 1? What would it be with a continuous dividend yield $q$?
2. In the time scan, why does the OTM call theta go to zero as $T \to 0$ while the ATM theta explodes?
3. Switch the pricer to `dividend_model="continuous"`: which greeks change, and why only those?
4. Check the American put against a binomial tree (CRR, 2000 steps) on the inputs left on the priced option
   (`spot_net`, `risk_free_rate`, `volatility`, `time_to_maturity`, `forward`).